# FinOps Starter — Backfill / Replay

Backfill semantics (see docs/operations_runbook.md):

* **Late-arriving data** is handled automatically by each source's `lookback_hours`.
* **Wider backfill window**: raise `lookback_hours` in `conf/sources/<source>.yml` —
  the merge is idempotent, re-reading is safe and cheap for unchanged records.
* **Full historical re-load** requires a checkpoint reset, which is DESTRUCTIVE and
  approval-gated; it is never performed automatically by this framework.

This notebook re-runs selected sources with the current (config-driven) windows.

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
dbutils.widgets.text("sources", "", "Comma-separated source ids to re-run (e.g. billing_usage,genie_usage)")

ENVIRONMENT = dbutils.widgets.get("environment")
SOURCES = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()] or None

In [0]:
import sys
from pathlib import Path

REPO = Path("/Workspace/Users/dhirajsardal@gmail.com/databricks-finops-starter")
sys.path.insert(0, str(REPO / "src"))

from finops.pipeline import run_pipeline  # noqa: E402

report = run_pipeline(
    spark,
    conf_root=REPO / "conf",
    environment=ENVIRONMENT,
    sources=SOURCES,
)
display(report)